In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [24]:
amazon = pd.read_csv("Amazon_Data.csv", low_memory=False)
target = "WeekTotal"
amazon = amazon.dropna(subset=[target])
amazon = amazon.sample(n=min(30000, amazon.shape[0]), random_state=25).reset_index(drop=True)

In [25]:
drop_cols = []
for col in amazon.columns:
    if col.lower() in ['order id', 'orderid', 'order_id', 'asin', 'sku']:
        drop_cols.append(col)
    elif amazon[col].dtype == "object" and amazon[col].nunique() / len(amazon) > 0.6:
        drop_cols.append(col)

In [26]:
leakage_cols = ['Week', 'Month-Year', 'Month', 'Year']
features = [c for c in amazon.columns if c not in drop_cols and c != target][:15]
features = [c for c in features if c not in leakage_cols]
X = amazon[features].copy()
y = amazon[target].astype(float)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25)

In [28]:
cat_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
num_features = X.select_dtypes(include=[np.number]).columns.tolist()

In [29]:
for col in cat_features:
    top_vals = X_train[col].astype(str).value_counts().nlargest(10).index
    X_train[col] = X_train[col].astype(str).apply(lambda x: x if x in top_vals else "Other")
    X_test[col] = X_test[col].astype(str).apply(lambda x: x if x in top_vals else "Other")

In [30]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features)
])

In [31]:
lr_model = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
rf_model = Pipeline([("prep", preprocessor), ("model", RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))])

In [32]:
lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
def evaluate(model, X_tr, X_te, y_tr, y_te):
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    return {
        "Train R2": r2_score(y_tr, pred_tr),
        "Test R2": r2_score(y_te, pred_te),
        "Test RMSE": mean_squared_error(y_te, pred_te),
        "Test MAE": mean_absolute_error(y_te, pred_te)
    }, pred_te

In [34]:
lr_metrics, lr_preds = evaluate(lr_model, X_train, X_test, y_train, y_test)
rf_metrics, rf_preds = evaluate(rf_model, X_train, X_test, y_train, y_test)

In [35]:
num_names = num_features
cat_names = list(rf_model.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(cat_features)) if cat_features else []
all_features = num_names + cat_names

In [36]:
importances = pd.DataFrame({
    "Feature": all_features,
    "Importance": rf_model.named_steps["model"].feature_importances_
}).sort_values("Importance", ascending=False).head(10)

In [37]:
print("\nLinear Regression:", lr_metrics)
print("\nRandom Forest:", rf_metrics)
print("\nTop Features:\n", importances)


Linear Regression: {'Train R2': 0.23481322615575084, 'Test R2': 0.2396191607209176, 'Test RMSE': 2810702.8318580044, 'Test MAE': 1195.7844588895705}

Random Forest: {'Train R2': 0.8737690875515638, 'Test R2': 0.17169643257669676, 'Test RMSE': 3061775.1820285926, 'Test MAE': 1167.2214459879751}

Top Features:
                                  Feature  Importance
2                       ship-postal-code    0.357751
1                                 Amount    0.264352
53                            Date_Other    0.152235
5                         Status_Pending    0.032926
10            Status_Shipped - Picked Up    0.024215
6   Status_Pending - Waiting for Pick Up    0.016490
27                       ship-city_Other    0.014879
36                      ship-state_Other    0.010211
3                       Status_Cancelled    0.009071
7                         Status_Shipped    0.007042


In [38]:
sample = X_test.head(10).copy()
sample["Actual"] = y_test.head(10).values
sample["Pred_LR"] = lr_preds[:10]
sample["Pred_RF"] = rf_preds[:10]
print("\nSample:\n", sample)


Sample:
                              Status Fulfilment Sales Channel  \
24155  Shipped - Delivered to Buyer   Merchant     Amazon.in   
23587                     Cancelled   Merchant     Amazon.in   
3644                        Shipped     Amazon     Amazon.in   
9884                        Shipped     Amazon     Amazon.in   
18817                       Shipped     Amazon     Amazon.in   
15266                       Shipped     Amazon     Amazon.in   
5336                      Cancelled   Merchant     Amazon.in   
25360                       Shipped     Amazon     Amazon.in   
12052                       Shipped     Amazon     Amazon.in   
29484  Shipped - Delivered to Buyer   Merchant     Amazon.in   

      ship-service-level  Qty  Amount  ship-city     ship-state  \
24155           Standard  1.0  735.00  BENGALURU      KARNATAKA   
23587           Standard  0.0  695.24      Other  UTTAR PRADESH   
3644           Expedited  1.0  383.00  HYDERABAD      TELANGANA   
9884           Ex

In [39]:
import json
with open("WeekTotal_Model_Summary.json", "w") as f:
    json.dump({"Target": target, "LR": lr_metrics, "RF": rf_metrics, "TopFeatures": importances.to_dict(orient="records")}, f, indent=2)